# IEEE-CIS Fraud Detection — Champion-Challenger Evaluation

## Goal
Validate the feature-selection decision made in `04_hyperparameter_tuning.ipynb`:
is `lightgbm_final.pkl` (champion, 412 features) a sound choice, or would the
full 443-feature config (challenger) have been meaningfully better?

## Why walk-forward validation?
A single split gives one point estimate of performance, with no idea how much
that estimate would vary on a different time window. Fraud patterns drift over
time, so we need to know: does any observed gap hold up across *multiple*
future periods, or was it a one-off?

## Plan
1. Combine train+val data, sort by `TransactionDT`, create expanding-window
   walk-forward folds (`TimeSeriesSplit`).
2. Train champion and challenger configs on each fold.
3. Compare AUC-PR mean/std across folds.
4. Compare profit (cost-matrix based) per fold.
5. Write up the promotion decision.

## Chapter 1 — Build Walk-Forward Folds

Combine train + val into one continuous timeline (sorted by `TransactionDT`),
then use `TimeSeriesSplit` to create expanding-window folds: each fold's
training set grows to include all prior data, and the validation set is the
next chunk of time.

In [7]:
import pandas as pd
import numpy as np
import json
import os
from sklearn.model_selection import TimeSeriesSplit

os.chdir('/Users/shaliqshukoor/fraud-risk-system')

train_df = pd.read_parquet('data/processed/train_features.parquet')
val_df   = pd.read_parquet('data/processed/val_features.parquet')

# Combine into one continuous timeline, sorted by transaction time
full_df = pd.concat([train_df, val_df], ignore_index=True)
full_df = full_df.sort_values('TransactionDT').reset_index(drop=True)

print(f'Combined shape: {full_df.shape}')
print(f'Time range: {full_df["TransactionDT"].min():,} to {full_df["TransactionDT"].max():,} seconds')
print(f'Total span: {(full_df["TransactionDT"].max() - full_df["TransactionDT"].min()) / 86400:.1f} days')
print(f'Overall fraud rate: {full_df["isFraud"].mean():.4f}')

Combined shape: (590540, 446)
Time range: 86,400 to 15,811,131 seconds
Total span: 182.0 days
Overall fraud rate: 0.0350


In [8]:
# Expanding-window walk-forward folds
tscv = TimeSeriesSplit(n_splits=4)

for fold, (train_idx, val_idx) in enumerate(tscv.split(full_df), start=1):
    train_fold = full_df.iloc[train_idx]
    val_fold   = full_df.iloc[val_idx]
    print(f'Fold {fold}:')
    print(f'  Train: {len(train_fold):>7,} rows | days {train_fold["TransactionDT"].min()/86400:6.1f} - {train_fold["TransactionDT"].max()/86400:6.1f} | fraud rate {train_fold["isFraud"].mean():.4f}')
    print(f'  Val:   {len(val_fold):>7,} rows | days {val_fold["TransactionDT"].min()/86400:6.1f} - {val_fold["TransactionDT"].max()/86400:6.1f} | fraud rate {val_fold["isFraud"].mean():.4f}')

Fold 1:
  Train: 118,108 rows | days    1.0 -   26.7 | fraud rate 0.0239
  Val:   118,108 rows | days   26.7 -   64.7 | fraud rate 0.0401
Fold 2:
  Train: 236,216 rows | days    1.0 -   64.7 | fraud rate 0.0320
  Val:   118,108 rows | days   64.7 -  101.2 | fraud rate 0.0375
Fold 3:
  Train: 354,324 rows | days    1.0 -  101.2 | fraud rate 0.0338
  Val:   118,108 rows | days  101.2 -  141.1 | fraud rate 0.0390
Fold 4:
  Train: 472,432 rows | days    1.0 -  141.1 | fraud rate 0.0351
  Val:   118,108 rows | days  141.1 -  183.0 | fraud rate 0.0344


## Chapter 2 — Train Champion vs Challenger on Each Fold

For each fold:
- Label-encode object columns (fit on that fold's train only — no leakage)
- Train **champion** (443 features) and **challenger** (412 selected features)
  using the same hyperparameters from the Optuna study — the only difference
  is the feature set
- Record AUC-PR on that fold's validation set

This isolates feature selection as the variable being tested.

In [12]:
import joblib
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import average_precision_score
from lightgbm import LGBMClassifier, early_stopping, log_evaluation

# Champion = our actual final model (lightgbm_final.pkl): 412 selected features
# Challenger = hypothetical alternative: full 443-feature config (pre-selection)
with open('data/processed/selected_feature_names.json') as f:
    champion_features = json.load(f)
with open('data/processed/feature_names.json') as f:
    challenger_features = json.load(f)

# Best hyperparameters from the saved Optuna study (04_hyperparameter_tuning.ipynb)
study = joblib.load('models/optuna_study_lgbm.pkl')
best_params = {
    **study.best_params,
    'n_estimators': 1000,
    'is_unbalance': True,
    'metric': 'average_precision',
    'random_state': 42,
    'n_jobs': -1,
    'verbose': -1
}
print(f'Loaded best params (AUC-PR {study.best_value:.4f}): {best_params}')

fold_results = []
predictions = {}  # (fold, model_name) -> {'y_val': ..., 'y_pred_proba': ...}

for fold, (train_idx, val_idx) in enumerate(tscv.split(full_df), start=1):
    train_fold = full_df.iloc[train_idx]
    val_fold   = full_df.iloc[val_idx]
    y_train_fold = train_fold['isFraud']
    y_val_fold   = val_fold['isFraud']

    for name, features in [('champion', champion_features), ('challenger', challenger_features)]:
        X_train_fold = train_fold[features].copy()
        X_val_fold   = val_fold[features].copy()

        # Label encode object columns, fit on this fold's train only
        obj_cols = X_train_fold.select_dtypes(include='object').columns.tolist()
        for col in obj_cols:
            le = LabelEncoder()
            X_train_fold[col] = le.fit_transform(X_train_fold[col].astype(str))
            X_val_fold[col]   = le.transform(X_val_fold[col].astype(str).map(
                lambda x: x if x in le.classes_ else le.classes_[0]
            ))

        model = LGBMClassifier(**best_params)
        model.fit(
            X_train_fold, y_train_fold,
            eval_set=[(X_val_fold, y_val_fold)],
            callbacks=[early_stopping(50, verbose=False), log_evaluation(-1)]
        )

        y_pred_proba = model.predict_proba(X_val_fold)[:, 1]
        auc_pr = average_precision_score(y_val_fold, y_pred_proba)

        predictions[(fold, name)] = {
            'y_val': y_val_fold.values,
            'y_pred_proba': y_pred_proba
        }

        fold_results.append({
            'fold': fold,
            'model': name,
            'n_features': len(features),
            'auc_pr': auc_pr,
            'best_iteration': model.best_iteration_
        })
        print(f'Fold {fold} | {name:>10} ({len(features)} feats) | AUC-PR: {auc_pr:.4f} | best_iter: {model.best_iteration_}')

results_df = pd.DataFrame(fold_results)
results_df

Loaded best params (AUC-PR 0.5495): {'learning_rate': 0.039802114210878925, 'num_leaves': 203, 'max_depth': 10, 'min_child_samples': 192, 'reg_lambda': 0.9771505975428012, 'n_estimators': 1000, 'is_unbalance': True, 'metric': 'average_precision', 'random_state': 42, 'n_jobs': -1, 'verbose': -1}
Fold 1 |   champion (412 feats) | AUC-PR: 0.5350 | best_iter: 276
Fold 1 | challenger (443 feats) | AUC-PR: 0.5328 | best_iter: 221
Fold 2 |   champion (412 feats) | AUC-PR: 0.5594 | best_iter: 419
Fold 2 | challenger (443 feats) | AUC-PR: 0.5612 | best_iter: 498
Fold 3 |   champion (412 feats) | AUC-PR: 0.5864 | best_iter: 651
Fold 3 | challenger (443 feats) | AUC-PR: 0.5841 | best_iter: 382
Fold 4 |   champion (412 feats) | AUC-PR: 0.5591 | best_iter: 720
Fold 4 | challenger (443 feats) | AUC-PR: 0.5630 | best_iter: 1000


,fold,model,n_features,auc_pr,best_iteration
0,1,champion,412,0.535004,276
1,1,challenger,443,0.532791,221
2,2,champion,412,0.559403,419
3,2,challenger,443,0.561225,498
4,3,champion,412,0.586351,651
5,3,challenger,443,0.584146,382
6,4,champion,412,0.559099,720
7,4,challenger,443,0.562966,1000


### Reading the Chapter 2 results

| Fold | Champion: lightgbm_final.pkl (412) | Challenger: full feature set (443) | Diff (challenger - champion) |
|---|---|---|---|
| 1 | 0.5350 | 0.5328 | -0.0022 |
| 2 | 0.5594 | 0.5612 | +0.0018 |
| 3 | 0.5864 | 0.5841 | -0.0023 |
| 4 | 0.5591 | 0.5630 | +0.0039 |

**Framing for this project**: `lightgbm_final.pkl` (412 features) is the
**champion** — it's the model we already tuned, selected, saved, and ran
explainability on. It's our actual current choice. The 443-feature config is
the **challenger** — a hypothetical "what if we'd kept all features instead of
dropping the 31 zero-importance ones?" — checked here to validate that
decision, not to propose replacing the champion.

**Observations:**
- The result is now a dead heat: champion wins folds 1 and 3, challenger wins
  folds 2 and 4. The differences (±0.002 to ±0.004) flip sign across folds —
  the classic signature of noise rather than a real effect.
- Fold-to-fold variance (champion: 0.535-0.586, challenger: 0.533-0.584 — both
  roughly a 0.05 swing) is still **an order of magnitude larger** than the
  champion/challenger gap.
- Fold 4's challenger hit `best_iteration=1000` — it used the full
  `n_estimators` budget without early stopping. Worth noting (could mean it's
  still improving, but the gain is marginal regardless), though it doesn't
  change the overall conclusion.

This confirms the 412 vs 443 feature difference is within noise — dropping the
31 zero-importance features for explainability cost effectively nothing in
predictive power. Chapter 3 puts a number on "within noise."

## Chapter 3 — Is the Difference Statistically Real?

For each fold, compute `challenger - champion` (a **paired** difference — same
fold, same data, only the feature set differs). Then:
- Mean and std of the per-fold AUC-PR for each model
- Mean and std of the paired differences
- A paired t-test: is the mean difference distinguishable from zero given the
  variance across folds?

In [10]:
from scipy import stats

# Mean / std per model across folds
summary = results_df.groupby('model')['auc_pr'].agg(['mean', 'std']).round(4)
print('--- AUC-PR across folds ---')
print(summary)

# Paired difference per fold (challenger - champion)
pivot = results_df.pivot(index='fold', columns='model', values='auc_pr')
pivot['diff'] = pivot['challenger'] - pivot['champion']
print('\n--- Per-fold comparison ---')
print(pivot)

print(f'\nMean diff (challenger - champion): {pivot["diff"].mean():.4f}')
print(f'Std of diff:                        {pivot["diff"].std():.4f}')

t_stat, p_value = stats.ttest_rel(pivot['challenger'], pivot['champion'])
print(f'\nPaired t-test: t={t_stat:.3f}, p={p_value:.3f}')
print('p > 0.05 -> cannot reject "no difference" -> the gap is not distinguishable from noise')

--- AUC-PR across folds ---
              mean     std
model                     
challenger  0.5603  0.0211
champion    0.5600  0.0210

--- Per-fold comparison ---
model  challenger  champion      diff
fold                                 
1        0.532791  0.535004 -0.002212
2        0.561225  0.559403  0.001821
3        0.584146  0.586351 -0.002204
4        0.562966  0.559099  0.003867

Mean diff (challenger - champion): 0.0003
Std of diff:                        0.0030

Paired t-test: t=0.210, p=0.847
p > 0.05 -> cannot reject "no difference" -> the gap is not distinguishable from noise


## Chapter 4 — Profit Comparison Per Fold

AUC-PR is a ranking metric — it doesn't tell us what happens at the actual
decision boundary we'd deploy with. Apply the **chosen operating threshold
(0.20)** from `04_threshold_optimization.ipynb` to both models' predictions in
each fold, and compare profit using the same cost matrix
(TP=+$200, FP=-$5, FN=-$250).

This answers the real question: "if we deployed either config with our chosen
threshold, which one makes more money, and is that consistent across time?"

In [13]:
from sklearn.metrics import confusion_matrix

with open('models/optimal_threshold.json') as f:
    optimal_threshold = json.load(f)['optimal_threshold']
print(f'Using threshold: {optimal_threshold}')

def calculate_profit(y_true, y_pred, tp_value=200, fp_cost=-5, fn_cost=-250):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    profit = (tp * tp_value) + (fp * fp_cost) + (fn * fn_cost)
    return profit, tp, fp, fn, tn

profit_results = []
for (fold, name), preds in predictions.items():
    y_pred = (preds['y_pred_proba'] >= optimal_threshold).astype(int)
    profit, tp, fp, fn, tn = calculate_profit(preds['y_val'], y_pred)
    profit_results.append({
        'fold': fold, 'model': name, 'profit': profit,
        'tp': tp, 'fp': fp, 'fn': fn
    })

profit_df = pd.DataFrame(profit_results).sort_values(['fold', 'model']).reset_index(drop=True)
print(profit_df.to_string(index=False))

pivot_profit = profit_df.pivot(index='fold', columns='model', values='profit')
pivot_profit['diff (challenger - champion)'] = pivot_profit['challenger'] - pivot_profit['champion']
print('\n--- Profit comparison ---')
print(pivot_profit)

Using threshold: 0.2
 fold      model  profit   tp    fp   fn
    1 challenger  393875 3791 25815  941
    1   champion  366765 3699 22957 1033
    2 challenger  348245 3446 18991  984
    2   champion  374570 3531 21376  899
    3 challenger  532775 4030 25595  581
    3   champion  470605 3810 18229  801
    4 challenger  341500 3186 15240  878
    4   champion  391050 3332 18470  732

--- Profit comparison ---
model  challenger  champion  diff (challenger - champion)
fold                                                     
1          393875    366765                         27110
2          348245    374570                        -26325
3          532775    470605                         62170
4          341500    391050                        -49550


## Chapter 5 — Promotion Decision

### Summary of evidence
- **AUC-PR (Chapter 3)**: champion (412 features) and challenger (443 features)
  are statistically indistinguishable across 4 walk-forward folds
  (mean diff = 0.0003, paired t-test p=0.847).
- **Profit at threshold=0.20 (Chapter 4)**: results swing wildly
  (-$49,550 to +$62,170 per fold), with no consistent winner — and the
  comparison itself is confounded by the fact that 0.20 was tuned for the
  champion specifically, so it isn't a true apples-to-apples operating point
  for the challenger.
- **Fold-to-fold variance dwarfs the champion/challenger gap** in both metrics —
  *when* you validate matters far more than *which* feature set you use.

### Decision: Keep the champion (`lightgbm_final.pkl`, 412 features)

No change is justified. The 443-feature alternative shows no statistically or
practically significant advantage over the 412-feature model that was already
selected, tuned, and explained via SHAP. Given that, the simpler model wins on
the tie-breaker criteria that actually matter operationally:

- **Explainability**: fewer features = simpler SHAP narratives for
  analysts/regulators (Phase 4).
- **Maintainability**: smaller feature pipeline, less to monitor for drift
  (Phase 6).
- **Inference cost**: marginally faster scoring in production (Phase 7).

### What this exercise demonstrated (the actual point of Phase 5)
Even though the decision didn't change, this walk-forward check was valuable —
it converted "we *think* the feature-selection tradeoff was small" (based on a
single split) into "we've *shown* the tradeoff is statistically indistinguishable
from zero across 4 independent time windows." That's the difference between an
assumption and a validated claim — exactly what a model-risk review would ask for.

### Next: Phase 6 — Monitoring
Now that there's a settled champion, Phase 6 (Evidently AI, PSI/drift detection)
is about detecting *when this champion starts to decay* in production — which is
where the "frozen artifact" version of champion-challenger (discussed earlier)
becomes relevant.